# OCSVM self-training 파이프라인 — 배치 입력 · pseudo-labeling · 반복 재학습 · 모니터링

위에서부터 순서대로 실행합니다. `DATASET`만 바꾸면 CN7·RG3 모두 같은 흐름으로 동작합니다.

| 단계 | 내용 |
|---|---|
| 1. 데이터 입력 | 비라벨 원본을 원본 인덱스 순서로 `BATCH_SIZE`행씩 가져옵니다. 지금은 CSV, 이후 DB 조회로 교체합니다(`fetch(offset, size)` 인터페이스 동일). |
| 2. 전처리 | 기존 보수적 전처리 규칙(24개 입력 검증, 결측·무한대 점검, 24개 값 완전 일치 패턴 통합)을 배치 단위로 적용합니다. 이전 배치에서 이미 본 패턴은 다시 라벨링하지 않습니다. |
| 3. 학습 | 라운드 0은 기존 파이프라인(`validation_f1_20260929` 선택 모델: 시나리오·nu·gamma·임계값)을 그대로 재현합니다. 이후 매 라운드 새 배치를 현재 모델로 predict하여 pseudo-label을 붙이고, 누적 pseudo 데이터를 포함해 재학습합니다. |
| 4. 결과 모니터링 | 라운드별 pseudo-label 구성·경보율, 라벨 개발 OOF·고정 test 성능, 점수 분포, PSI, pseudo-label 뒤집힘 비율, 변수 평균 이동, PCA 시각화를 저장합니다. |

**OCSVM 재학습 규칙**: OCSVM은 정상만으로 학습하므로 재학습 입력은 `라벨 개발 정상 + 누적 pseudo-정상`입니다. pseudo-불량은 기록·모니터링하며 학습 입력에는 넣지 않습니다. 하이퍼파라미터와 임계값은 기존 선택값을 고정합니다.

**평가 규칙**: 라벨 test는 어떤 학습·선택에도 쓰지 않습니다. 라운드별 개발 OOF는 각 fold의 학습 정상에 누적 pseudo-정상을 더해 적합하고, 해당 fold validation에 같은 임계값을 적용합니다.

## 0. 실행 설정

배치 크기(`BATCH_SIZE`)는 논의 대상이므로 여기서만 바꿉니다. `MAX_ROUNDS=None`이면 비라벨 원본을 끝까지 소진합니다.

In [ ]:
DATASET = 'cn7'            # 'cn7' 또는 'rg3'
BATCH_SIZE = 5000          # 1회 입력 단위(원본 행 수) — 논의 대상
MAX_ROUNDS = None          # None이면 원본 소진까지
CONFIDENCE_MARGIN = 0.0    # pseudo-정상 채택 기준: score <= threshold - margin (0이면 예측 0 전체 채택)
RELABEL_POOL = False       # True면 매 라운드 새 모델로 기존 pseudo-label을 다시 붙임(기본: 최초 라벨 고정)
SEED = 42
RUN_ID = f'bs{BATCH_SIZE}_m{CONFIDENCE_MARGIN:g}' + ('_relabel' if RELABEL_POOL else '')

### 0.1 경로와 라이브러리

In [ ]:
from pathlib import Path
import os, json, hashlib, platform, time
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/origin').is_dir())
os.environ.setdefault('MPLCONFIGDIR', str(ROOT/'tmp/matplotlib_cache'))
import numpy as np
import pandas as pd
import sklearn
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Image
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

_names = {f.name for f in font_manager.fontManager.ttflist}
for _font in ['Malgun Gothic', 'NanumGothic', 'AppleGothic', 'Noto Sans CJK KR', 'Noto Sans CJK JP']:
    if _font in _names:
        plt.rcParams['font.family'] = _font
        break
plt.rcParams['axes.unicode_minus'] = False

MODEL_DIR = ROOT/f'output/ocsvm_{DATASET}_integrated/validation_f1_20260929'
DATA_DIR = ROOT/f'data/processed/{DATASET}/conservative'
RAW_UNLABELED = ROOT/f'data/origin/moldset_unlabeled_{DATASET}.csv'
OUT = ROOT/f'output/self_training/{DATASET}/{RUN_ID}'
(OUT/'models').mkdir(parents=True, exist_ok=True)


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def write_json(path, record):
    Path(path).write_text(json.dumps(record, ensure_ascii=False, indent=2, default=float), encoding='utf-8')

print('데이터:', DATASET, '| 배치 크기:', BATCH_SIZE, '| 결과:', OUT.relative_to(ROOT))

### 0.2 기존 파이프라인 정의(통합 노트북 A.2·B.1과 동일)

기존 선택 모델을 그대로 재현하기 위해 공정 가설·파생변수·변환 함수를 통합 노트북에서 옮겼습니다. CN7의 `legacy_without_cycle`과 RG3의 `without_cycle`은 같은 규칙(형체·주기 변수 제외)입니다.

In [ ]:
GROUPS = {
    '충전과 전환': ['Injection_Time', 'Filling_Time', 'Cushion_Position',
                'Max_Injection_Speed', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure'],
    '계량과 가소화': ['Plasticizing_Time', 'Plasticizing_Position', 'Max_Screw_RPM',
                 'Average_Screw_RPM', 'Max_Back_Pressure', 'Average_Back_Pressure'],
    '실린더와 호퍼 온도': [f'Barrel_Temperature_{i}' for i in range(1, 7)] + ['Hopper_Temperature'],
    '금형 온도': ['Mold_Temperature_3', 'Mold_Temperature_4'],
    '형체와 전체 주기': ['Clamp_Close_Time', 'Clamp_Open_Position', 'Cycle_Time'],
}
SCENARIOS = {
    'raw': (), 'legacy_without_cycle': (), 'without_cycle': (),
    'pressure': ('pressure',), 'plasticizing': ('plasticizing',), 'thermal': ('thermal',),
    'domain_all': ('pressure', 'plasticizing', 'thermal'), 'group_pca': (),
}
WITHOUT_CYCLE = {'legacy_without_cycle', 'without_cycle'}
DOMAIN_SPEC = [
    dict(feature='사출_전환압력_상대편차', group='pressure', operation='difference',
         columns=['Max_Injection_Pressure', 'Max_Switch_Over_Pressure']),
    dict(feature='스크루RPM_상대편차', group='plasticizing', operation='difference',
         columns=['Max_Screw_RPM', 'Average_Screw_RPM']),
    dict(feature='배압_상대편차', group='plasticizing', operation='difference',
         columns=['Max_Back_Pressure', 'Average_Back_Pressure']),
    dict(feature='배럴_상대수준평균', group='thermal', operation='mean',
         columns=[f'Barrel_Temperature_{i}' for i in range(1, 7)]),
    dict(feature='배럴_상대편차표준편차', group='thermal', operation='std',
         columns=[f'Barrel_Temperature_{i}' for i in range(1, 7)]),
    dict(feature='금형_상대수준평균', group='thermal', operation='mean',
         columns=['Mold_Temperature_3', 'Mold_Temperature_4']),
    dict(feature='금형온도_상대편차', group='thermal', operation='difference',
         columns=['Mold_Temperature_3', 'Mold_Temperature_4']),
]
FEATURES = sum(GROUPS.values(), [])


def derived_frame(z, specs):
    result = pd.DataFrame(index=z.index)
    for spec in specs:
        values = z[spec['active_columns']]
        if spec['operation'] == 'difference':
            result[spec['feature']] = values.iloc[:, 0] - values.iloc[:, 1]
        elif spec['operation'] == 'mean':
            result[spec['feature']] = values.mean(axis=1)
        elif spec['operation'] == 'std':
            result[spec['feature']] = values.std(axis=1, ddof=0)
        else:
            raise ValueError(spec['operation'])
    return result


def unscaled_frame(bundle, inputs):
    inputs = inputs[bundle['input_columns']]
    z = pd.DataFrame(bundle['normal_scaler'].transform(inputs), index=inputs.index, columns=inputs.columns)
    if bundle['scenario'] == 'group_pca':
        frame = pd.DataFrame(index=inputs.index)
        for group, item in bundle['pcas'].items():
            values = z[item['columns']].to_numpy()
            scores = item['pca'].transform(values)
            for j in range(scores.shape[1]):
                frame[f'{group}_PC{j+1}'] = scores[:, j]
            if len(item['columns']) > scores.shape[1]:
                frame[f'{group}_재구성RMSE'] = np.sqrt(np.mean((values-item['pca'].inverse_transform(scores))**2, axis=1))
        return frame
    return pd.concat([z[bundle['base_columns']], derived_frame(z, bundle['specs'])], axis=1)


def fit_features(normal, scenario):
    if scenario not in SCENARIOS:
        raise ValueError(scenario)
    active = [c for c in normal if normal[c].nunique() > 1]
    scaler = StandardScaler().fit(normal)
    bundle = dict(scenario=scenario, input_columns=list(normal), normal_scaler=scaler,
                  specs=[], skipped_specs=[], pcas={})
    for spec in DOMAIN_SPEC:
        if spec['group'] not in SCENARIOS[scenario]:
            continue
        cols = [c for c in spec['columns'] if c in active]
        if len(cols) < 2:
            bundle['skipped_specs'].append({'feature': spec['feature'], 'reason': '활성 센서 2개 미만'})
        else:
            bundle['specs'].append({**spec, 'active_columns': cols})
    bundle['base_columns'] = [c for c in active if scenario not in WITHOUT_CYCLE
                              or c not in GROUPS['형체와 전체 주기']]
    if scenario == 'group_pca':
        z = pd.DataFrame(scaler.transform(normal), columns=normal.columns, index=normal.index)
        for group, cols in GROUPS.items():
            cols = [c for c in cols if c in active]
            if cols:
                bundle['pcas'][group] = {'columns': cols, 'pca': PCA(n_components=min(2, len(cols)),
                                                  svd_solver='full').fit(z[cols].to_numpy())}
    frame = unscaled_frame(bundle, normal)
    bundle['variance'] = VarianceThreshold(0).fit(frame)
    values = bundle['variance'].transform(frame)
    bundle['feature_scaler'] = StandardScaler().fit(values)
    bundle['feature_names'] = list(frame.columns[bundle['variance'].get_support()])
    return bundle, transform_features(bundle, normal)


def transform_features(bundle, inputs):
    frame = unscaled_frame(bundle, inputs)
    values = bundle['feature_scaler'].transform(bundle['variance'].transform(frame))
    assert np.isfinite(values).all()
    return values


def fit_model(values, nu, multiplier):
    model = OneClassSVM(kernel='rbf', nu=float(nu), gamma=float(multiplier)/values.shape[1],
                        tol=1e-4, max_iter=100000).fit(values)
    assert model.fit_status_ == 0, 'OCSVM 수렴 실패'
    return model


def classification_metrics(truth, prediction):
    truth, prediction = np.asarray(truth), np.asarray(prediction)
    tp = int(((truth == 1) & (prediction == 1)).sum()); fn = int(((truth == 1) & (prediction == 0)).sum())
    fp = int(((truth == 0) & (prediction == 1)).sum()); tn = int(((truth == 0) & (prediction == 0)).sum())
    return dict(TP=tp, FN=fn, FP=fp, TN=tn, recall=tp/(tp+fn) if tp+fn else 0.,
                FPR=fp/(fp+tn) if fp+tn else 0., precision=tp/(tp+fp) if tp+fp else 0.,
                F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.)

## 1. 데이터 입력

### 1.1 라벨 데이터(기존 전처리·고정 분할 결과)

기존 보수적 전처리 결과와 고정 test·개발 4-fold 매핑을 해시로 확인한 뒤 불러옵니다.

In [ ]:
split_manifest = json.loads((DATA_DIR/'splits/split_manifest.json').read_text(encoding='utf-8'))
for filename, expected in split_manifest['source_sha256'].items():
    assert digest(DATA_DIR/filename) == expected, f'전처리 결과 변경: {filename}'
assert digest(DATA_DIR/'splits/split_assignments.csv') == split_manifest['artifact_sha256']['split_assignments.csv']
X = pd.read_csv(DATA_DIR/'X_labeled.csv', float_precision='round_trip')
y = pd.read_csv(DATA_DIR/'y_labeled.csv').PassOrFail
assignment = pd.read_csv(DATA_DIR/'splits/split_assignments.csv')
assert list(X.columns) == FEATURES or sorted(X.columns) == sorted(FEATURES)
X = X[FEATURES] if list(X.columns) != FEATURES else X
assert np.array_equal(assignment.pattern_row, np.arange(len(X))) and np.array_equal(assignment.label, y)
dev = assignment.index[assignment.partition.eq('development')].to_numpy()
test = assignment.index[assignment.partition.eq('test')].to_numpy()
fold_ids = assignment.loc[dev, 'cv_fold']
Xdev, ydev, Xtest, ytest = X.loc[dev], y.loc[dev], X.loc[test], y.loc[test]
dev_normal = Xdev.index[ydev.eq(0)]
print(f'라벨 패턴 {len(X)}개 | 개발 {len(dev)}개(위험 {int(ydev.sum())}) | test {len(test)}개(위험 {int(ytest.sum())})')

### 1.2 비라벨 배치 입력기

`fetch(offset, size)`는 원본 인덱스 오름차순으로 `size`행을 돌려줍니다. DB 연결 시 같은 시그니처로 `ORDER BY source_id LIMIT size OFFSET offset` 조회로 교체하면 이후 단계는 바뀌지 않습니다.

In [ ]:
class CsvBatchSource:
    """CSV를 DB 테이블처럼 사용하는 배치 입력기."""
    def __init__(self, path):
        self.path = Path(path)
        frame = pd.read_csv(self.path, float_precision='round_trip').rename(columns={'Unnamed: 0': 'source_id'})
        assert frame.source_id.is_unique
        self._table = frame.sort_values('source_id', kind='stable').reset_index(drop=True)
        self.sha256 = digest(self.path)

    def __len__(self):
        return len(self._table)

    def fetch(self, offset, size):
        return self._table.iloc[offset:offset+size].copy()


source = CsvBatchSource(RAW_UNLABELED)
n_rounds = int(np.ceil(len(source)/BATCH_SIZE))
if MAX_ROUNDS is not None:
    n_rounds = min(n_rounds, MAX_ROUNDS)
print(f'비라벨 원본 {len(source):,}행 → {n_rounds}개 배치(배치당 최대 {BATCH_SIZE:,}행)')

## 2. 전처리(기존 규칙의 배치 적용)

기존 전처리와 같은 점검을 배치마다 수행합니다: 24개 입력 존재·숫자형·결측/무한대 없음 → 24개 값이 완전히 같은 행을 하나의 패턴으로 통합(첫 행 대표, 원본 ID 목록 보존). 이전 배치에서 이미 라벨링한 패턴과 라벨 데이터에 있는 패턴은 새로 라벨링하지 않고 개수만 기록합니다. 스케일링은 학습 단계에서 정상 학습 행으로만 적합합니다.

In [ ]:
class PatternRegistry:
    """배치를 넘어 패턴 ID를 전역으로 유지합니다."""
    def __init__(self, prefix, labeled_inputs):
        self.prefix, self.ids = prefix, {}
        self.labeled_keys = set(map(tuple, labeled_inputs.to_numpy()))

    def register(self, key):
        if key not in self.ids:
            self.ids[key] = f'{self.prefix}_{len(self.ids):06d}'
        return self.ids[key]


def preprocess_batch(raw, registry):
    missing = [c for c in FEATURES if c not in raw.columns]
    assert not missing, f'입력 컬럼 누락: {missing}'
    x = raw[FEATURES]
    assert all(pd.api.types.is_numeric_dtype(x[c]) for c in FEATURES)
    assert np.isfinite(x.to_numpy()).all(), '결측값 또는 무한대 발견'
    grouped = raw.groupby(FEATURES, sort=False, dropna=False)
    patterns = grouped.agg(total_count=('source_id', 'size'), representative_source_id=('source_id', 'first'),
                           source_ids=('source_id', lambda s: json.dumps(s.tolist()))).reset_index()
    keys = list(map(tuple, patterns[FEATURES].to_numpy()))
    seen = np.array([k in registry.ids for k in keys], dtype=bool)
    in_labeled = np.array([k in registry.labeled_keys for k in keys], dtype=bool)
    new = patterns.loc[~seen & ~in_labeled].copy()
    new.insert(0, 'pattern_id', [registry.register(k) for k in map(tuple, new[FEATURES].to_numpy())])
    quality = dict(batch_rows=len(raw), batch_patterns=len(patterns), within_batch_duplicate_rows=len(raw)-len(patterns),
                   already_seen_patterns=int(seen.sum()), labeled_overlap_patterns=int(in_labeled.sum()),
                   new_patterns=len(new),
                   source_id_min=int(raw.source_id.min()), source_id_max=int(raw.source_id.max()))
    return new.reset_index(drop=True), quality

registry = PatternRegistry(f'{DATASET.upper()}_U', X)

## 3. 학습

### 3.1 라운드 0 — 기존 파이프라인 재현

기존 선택(`selection_manifest.json`)의 시나리오·nu·gamma·임계값으로 개발 정상 전체를 적합합니다. 저장된 기존 모델과 test 점수가 같은지, 개발 OOF 혼동행렬이 기존 선택 기록과 같은지 확인합니다.

In [ ]:
selection = json.loads((MODEL_DIR/'selection_manifest.json').read_text(encoding='utf-8'))
choice = selection['choices'][0]
assert choice['status'] == 'selected_for_followup'
SCENARIO, NU, GAMMA_MULT, THRESHOLD = choice['scenario'], float(choice['nu']), float(choice['gamma_multiplier']), float(choice['threshold'])
print(f'기존 선택: {SCENARIO}, nu={NU:g}, gamma 배수={GAMMA_MULT:.6g}, t={THRESHOLD:.6g}')


def train(normal_inputs):
    bundle, values = fit_features(normal_inputs, SCENARIO)
    return dict(features=bundle, model=fit_model(values, NU, GAMMA_MULT), threshold=THRESHOLD)


def score(artifact, inputs):
    return -artifact['model'].decision_function(transform_features(artifact['features'], inputs[FEATURES]))


def labeled_evaluation(pseudo_normal):
    """개발 OOF(각 fold 정상 + pseudo-정상) 및 고정 test(개발 정상 + pseudo-정상)."""
    truth, pred = [], []
    for fold in sorted(fold_ids.unique()):
        valid = Xdev.index[fold_ids.eq(fold)]
        fit_idx = Xdev.index[fold_ids.ne(fold) & ydev.eq(0)]
        art = train(pd.concat([Xdev.loc[fit_idx], pseudo_normal], ignore_index=True))
        truth.append(ydev.loc[valid].to_numpy()); pred.append((score(art, Xdev.loc[valid]) > THRESHOLD).astype(int))
    oof = classification_metrics(np.concatenate(truth), np.concatenate(pred))
    final = train(pd.concat([Xdev.loc[dev_normal], pseudo_normal], ignore_index=True))
    test_scores = score(final, Xtest)
    test_metrics = classification_metrics(ytest, (test_scores > THRESHOLD).astype(int))
    return final, oof, test_metrics, test_scores

EMPTY = pd.DataFrame(columns=FEATURES, dtype=float)
model, oof0, test0, test_scores0 = labeled_evaluation(EMPTY)
saved = joblib.load(MODEL_DIR/'f1_selected_model.joblib')
saved_scores = -saved['model'].decision_function(transform_features(saved['features'], Xtest))
assert np.allclose(test_scores0, saved_scores, rtol=0, atol=1e-8), '기존 모델 재현 실패'
assert all(oof0[k] == int(choice[k]) for k in ['TP', 'FP', 'FN', 'TN']), '기존 OOF 재현 실패'
print('라운드 0 재현 확인: 기존 모델과 test 점수 일치, 개발 OOF 혼동행렬 일치')
print('개발 OOF:', {k: round(v, 4) for k, v in oof0.items()})
print('고정 test:', {k: round(v, 4) for k, v in test0.items()})

### 3.2 반복 self-training

각 라운드: ① 배치 입력 → ② 전처리 → ③ 현재 모델로 predict하여 pseudo-label 부여(점수 > t이면 1) → ④ 누적 pseudo-정상을 포함해 재학습 → ⑤ 라벨 개발 OOF·고정 test 평가 → ⑥ 기존 pseudo 데이터를 새 모델로 다시 채점해 라벨 뒤집힘 비율 기록. 라운드별 모델은 `models/`에 저장합니다.

In [ ]:
ref_scores = score(model, Xdev.loc[dev_normal])   # 라운드 0 모델의 라벨 정상 점수(시각화 기준)
dev_normal_mean = Xdev.loc[dev_normal].mean()


def psi(reference, current, bins=10):
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins+1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(reference, edges)[0]/len(reference); c = np.histogram(current, edges)[0]/len(current)
    r, c = np.clip(r, 1e-6, None), np.clip(c, 1e-6, None)
    return float(np.sum((c-r)*np.log(c/r)))

pool = pd.DataFrame(columns=['pattern_id', 'round', *FEATURES, 'total_count', 'representative_source_id',
                             'source_ids', 'score_at_label', 'pseudo_label', 'current_label'])
rounds = [dict(round=0, **{f'oof_{k}': v for k, v in oof0.items()}, **{f'test_{k}': v for k, v in test0.items()},
               pool_patterns=0, pool_pseudo_normal=0, pool_pseudo_defect=0, train_normals=len(dev_normal))]
drift_rows, batch_scores = [], {}
joblib.dump({**model, 'round': 0, 'choice': choice}, OUT/'models/round_00.joblib')

for r in range(1, n_rounds+1):
    start = time.time()
    raw = source.fetch((r-1)*BATCH_SIZE, BATCH_SIZE)
    new, quality = preprocess_batch(raw, registry)                                  # 2. 전처리
    ref_now = score(model, Xdev.loc[dev_normal])                                    # 현재 모델 기준 라벨 정상 점수
    s = score(model, new) if len(new) else np.array([])                             # 3. predict
    labels = (s > THRESHOLD).astype(int)
    adopt = (labels == 0) & (s <= THRESHOLD - CONFIDENCE_MARGIN)
    new = new.assign(round=r, score_at_label=s, pseudo_label=labels, current_label=labels,
                     adopted_normal=adopt)
    batch_scores[r] = s
    pool = pd.concat([pool, new], ignore_index=True)
    pseudo_normal = pool.loc[pool.current_label.eq(0) & pool.adopted_normal.astype(bool), FEATURES].astype(float)
    model, oof, test_m, _ = labeled_evaluation(pseudo_normal)                      # 재학습 + 평가
    pool_scores = score(model, pool[FEATURES].astype(float))
    relabel = (pool_scores > THRESHOLD).astype(int)
    flip = float((relabel != pool.current_label.to_numpy()).mean()) if len(pool) else 0.
    if RELABEL_POOL:
        pool['current_label'] = relabel
        pool['adopted_normal'] = (relabel == 0) & (pool_scores <= THRESHOLD - CONFIDENCE_MARGIN)
    joblib.dump({**model, 'round': r, 'choice': choice}, OUT/f'models/round_{r:02d}.joblib')
    drift_rows.append({'round': r, **(new[FEATURES].astype(float).mean() - dev_normal_mean).to_dict()} if len(new) else {'round': r})
    rounds.append(dict(round=r, **quality, pseudo_normal=int((labels == 0).sum()), pseudo_defect=int(labels.sum()),
        adopted_normal=int(adopt.sum()), alert_rate=float(labels.mean()) if len(labels) else np.nan,
        score_median=float(np.median(s)) if len(s) else np.nan, score_psi=psi(ref_now, s) if len(s) else np.nan,
        pool_patterns=len(pool), pool_pseudo_normal=int(pool.current_label.eq(0).sum()),
        pool_pseudo_defect=int(pool.current_label.eq(1).sum()), train_normals=len(dev_normal)+len(pseudo_normal),
        pool_flip_rate=flip, **{f'oof_{k}': v for k, v in oof.items()}, **{f'test_{k}': v for k, v in test_m.items()},
        seconds=round(time.time()-start, 2)))
    print(f"R{r:02d} 입력 {quality['batch_rows']:,}행 → 신규 패턴 {len(new):,} | pseudo 정상 {int((labels==0).sum()):,} / 불량 {int(labels.sum()):,}"
          f" | 경보율 {rounds[-1]['alert_rate']:.3f} | OOF F1 {oof['F1']:.3f} | test F1 {test_m['F1']:.3f} | 뒤집힘 {flip:.3f}")

round_metrics = pd.DataFrame(rounds)
round_metrics.to_csv(OUT/'round_metrics.csv', index=False)
pool.drop(columns=FEATURES).to_csv(OUT/'pseudo_labels.csv', index=False)
drift = pd.DataFrame(drift_rows).set_index('round')
drift.to_csv(OUT/'batch_feature_drift.csv')

### 3.3 전처리 일관성 검증

원본을 모두 소진한 경우, 배치 단위로 통합한 비라벨 패턴 집합이 기존 전처리의 `X_unlabeled.csv`와 같은지 확인합니다.

In [ ]:
consumed = sum(q.get('batch_rows', 0) for q in rounds[1:])
reference_path = DATA_DIR/'X_unlabeled.csv'
if consumed == len(source) and reference_path.exists():
    reference = pd.read_csv(reference_path, float_precision='round_trip')[FEATURES]
    batch_keys = set(registry.ids) | {k for k in map(tuple, reference.to_numpy()) if k in registry.labeled_keys}
    assert set(map(tuple, reference.to_numpy())) == batch_keys, '배치 전처리 결과가 기존 전처리와 다릅니다.'
    print(f'확인: 배치 통합 패턴 {len(registry.ids):,}개 = 기존 비라벨 전처리 {len(reference):,}개'
          + (f' (라벨과 겹치는 {len(batch_keys)-len(registry.ids)}개 제외)' if len(batch_keys) != len(registry.ids) else ''))
else:
    print(f'원본 {consumed:,}/{len(source):,}행 처리 — 전체 비교는 원본 소진 시에만 수행')

## 4. 결과 모니터링

### 4.1 라운드별 지표 표

In [ ]:
view = round_metrics[['round', 'batch_rows', 'new_patterns', 'pseudo_normal', 'pseudo_defect', 'alert_rate',
    'score_psi', 'pool_flip_rate', 'train_normals', 'oof_F1', 'oof_recall', 'oof_FPR', 'oof_TP', 'oof_FP',
    'test_F1', 'test_TP', 'test_FP', 'test_FN']].copy()
display(view.round(4))

### 4.2 pseudo-label 구성과 경보율

In [ ]:
m = round_metrics.loc[round_metrics['round'] > 0]
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(m['round'], m.pseudo_normal, color='#4C78A8', label='pseudo 정상(0)')
ax.bar(m['round'], m.pseudo_defect, bottom=m.pseudo_normal, color='#E45756', label='pseudo 불량(1)')
ax.set(xlabel='라운드', ylabel='신규 패턴 수', title=f'{DATASET.upper()} 라운드별 pseudo-label 구성')
ax2 = ax.twinx(); ax2.plot(m['round'], m.alert_rate, 'k-o', label='경보율')
ax2.axhline(float(y.loc[dev].mean()), color='gray', ls='--', label='라벨 개발 위험 비율')
ax2.set(ylabel='경보율', ylim=(0, max(1e-3, float(m.alert_rate.max())*1.15)))
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1+h2, l1+l2, loc='upper left'); fig.tight_layout()
fig.savefig(OUT/'01_pseudo_label_composition.png', dpi=130); plt.close(fig)
display(Image(filename=str(OUT/'01_pseudo_label_composition.png')))

### 4.3 라벨 데이터 성능 추이

라운드 0이 기존 파이프라인 기준선입니다. test 불량 수가 적어 1건 차이로도 F1이 크게 변하므로 OOF와 함께 봅니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharex=True)
for key, style in [('F1', '-o'), ('recall', '-s'), ('precision', '-^')]:
    axes[0].plot(round_metrics['round'], round_metrics[f'oof_{key}'], style, label=f'OOF {key}')
    axes[1].plot(round_metrics['round'], round_metrics[f'test_{key}'], style, label=f'test {key}')
for ax, name in zip(axes, ['개발 OOF', '고정 test']):
    ax.set(xlabel='라운드', ylim=(-0.02, 1.02), title=f'{name} 판정 성능'); ax.legend(); ax.grid(alpha=.3)
axb = axes[0].twinx(); axb.plot(round_metrics['round'], round_metrics.oof_FPR, 'r--', label='OOF FPR'); axb.set_ylabel('FPR', color='r')
fig.tight_layout(); fig.savefig(OUT/'02_labeled_performance.png', dpi=130); plt.close(fig)
display(Image(filename=str(OUT/'02_labeled_performance.png')))

### 4.4 점수 분포와 안정성

왼쪽: 각 배치를 라벨링할 때의 위험 점수 분포(점선 = 임계값 t, 회색 = 라운드 0 모델의 라벨 개발 정상 점수). 오른쪽: 같은 모델로 채점한 라벨 정상 점수 대비 배치 점수 PSI와 누적 pseudo-label 뒤집힘 비율.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={'width_ratios': [2, 1]})
data = [ref_scores] + [batch_scores[r] for r in sorted(batch_scores) if len(batch_scores[r])]
names = ['라벨 정상'] + [f'R{r}' for r in sorted(batch_scores) if len(batch_scores[r])]
bp = axes[0].boxplot(data, showfliers=False, patch_artist=True)
bp['boxes'][0].set_facecolor('lightgray')
axes[0].set_xticks(range(1, len(names)+1), names, rotation=45)
axes[0].axhline(THRESHOLD, color='red', ls='--', label=f't={THRESHOLD:.4g}')
axes[0].set(ylabel='위험 점수(-decision_function)', title='배치별 점수 분포'); axes[0].legend()
axes[1].plot(m['round'], m.score_psi, '-o', label='점수 PSI')
axes[1].axhline(0.25, color='gray', ls=':', label='PSI 0.25(큰 이동 관례)')
axes[1].plot(m['round'], m.pool_flip_rate, '-s', label='pseudo-label 뒤집힘 비율')
axes[1].set(xlabel='라운드', title='분포 이동·라벨 안정성'); axes[1].legend(); axes[1].grid(alpha=.3)
fig.tight_layout(); fig.savefig(OUT/'03_score_stability.png', dpi=130); plt.close(fig)
display(Image(filename=str(OUT/'03_score_stability.png')))

### 4.5 라벨링 결과 시각화 — PCA

라벨 개발 정상으로 표준화·PCA를 적합하고 pseudo 데이터(최대 6,000개 표본)와 라벨 데이터를 같은 2차원 좌표에 표시합니다.

In [ ]:
scaler_viz = StandardScaler().fit(Xdev.loc[dev_normal])
pca_viz = PCA(n_components=2, random_state=SEED).fit(scaler_viz.transform(Xdev.loc[dev_normal]))
proj = lambda frame: pca_viz.transform(scaler_viz.transform(frame[FEATURES].astype(float)))
sample = pool.sample(min(6000, len(pool)), random_state=SEED) if len(pool) else pool
fig, ax = plt.subplots(figsize=(9, 7))
for label, color, name in [(0, '#4C78A8', 'pseudo 정상'), (1, '#E45756', 'pseudo 불량')]:
    part = sample.loc[sample.current_label.eq(label)]
    if len(part):
        p = proj(part); ax.scatter(p[:, 0], p[:, 1], s=5, alpha=.25, color=color, label=f'{name} ({int(pool.current_label.eq(label).sum()):,})')
p = proj(X.loc[y.eq(0)]); ax.scatter(p[:, 0], p[:, 1], s=14, facecolors='none', edgecolors='black', lw=.6, label='라벨 정상')
p = proj(X.loc[y.eq(1)]); ax.scatter(p[:, 0], p[:, 1], s=70, marker='X', color='gold', edgecolors='black', label='라벨 위험')
ax.set(xlabel=f'PC1 ({pca_viz.explained_variance_ratio_[0]:.1%})', ylabel=f'PC2 ({pca_viz.explained_variance_ratio_[1]:.1%})',
       title=f'{DATASET.upper()} pseudo-label 결과(최종 라벨) — 라벨 정상 기준 PCA')
ax.legend(markerscale=2); fig.tight_layout(); fig.savefig(OUT/'04_pseudo_label_pca.png', dpi=130); plt.close(fig)
display(Image(filename=str(OUT/'04_pseudo_label_pca.png')))

### 4.6 배치별 입력 변수 평균 이동

각 배치 신규 패턴의 변수 평균 − 라벨 개발 정상 평균(제공 표준화 단위).

In [ ]:
if len(drift.columns):
    fig, ax = plt.subplots(figsize=(13, 0.35*len(drift)+3))
    lim = float(np.nanmax(np.abs(drift.to_numpy()))) or 1.
    im = ax.imshow(drift.to_numpy(), aspect='auto', cmap='RdBu_r', vmin=-lim, vmax=lim)
    ax.set_xticks(range(len(drift.columns)), drift.columns, rotation=75, fontsize=8)
    ax.set_yticks(range(len(drift)), [f'R{r}' for r in drift.index])
    fig.colorbar(im, ax=ax, label='평균 차이'); ax.set_title('배치 평균 − 라벨 개발 정상 평균')
    fig.tight_layout(); fig.savefig(OUT/'05_feature_drift.png', dpi=130); plt.close(fig)
    display(Image(filename=str(OUT/'05_feature_drift.png')))

### 4.7 실행 기록 저장

In [ ]:
first, last = round_metrics.iloc[0], round_metrics.iloc[-1]
summary = f"""# {DATASET.upper()} OCSVM self-training 결과 ({RUN_ID})

- 기존 선택 고정: {SCENARIO}, nu={NU:g}, gamma 배수={GAMMA_MULT:.6g}, t={THRESHOLD:.6g}
- 배치 {BATCH_SIZE:,}행 × {n_rounds}라운드, 비라벨 원본 {consumed:,}/{len(source):,}행 처리
- 누적 pseudo 패턴 {len(pool):,}개: 정상 {int(pool.current_label.eq(0).sum()):,} / 불량 {int(pool.current_label.eq(1).sum()):,}
- 전체 경보율 {pool.current_label.mean() if len(pool) else float('nan'):.4f} (라벨 개발 위험 비율 {float(y.loc[dev].mean()):.4f})
- 학습 정상 수: {int(first.train_normals):,} → {int(last.train_normals):,}
- 개발 OOF F1: {first.oof_F1:.4f} → {last.oof_F1:.4f} (TP {int(first.oof_TP)}→{int(last.oof_TP)}, FP {int(first.oof_FP)}→{int(last.oof_FP)})
- 고정 test F1: {first.test_F1:.4f} → {last.test_F1:.4f} (TP {int(first.test_TP)}→{int(last.test_TP)}, FP {int(first.test_FP)}→{int(last.test_FP)}, FN {int(first.test_FN)}→{int(last.test_FN)})
- 마지막 라운드 pseudo-label 뒤집힘 비율 {last.get('pool_flip_rate', float('nan')):.4f}

test는 선택·재학습에 사용하지 않았습니다. 라벨·비라벨 원본은 파일별로 따로 표준화되어 있어 좌표계 정합은 확인되지 않았습니다.
"""
(OUT/'analysis_summary.md').write_text(summary, encoding='utf-8'); print(summary)
write_json(OUT/'run_config.json', dict(dataset=DATASET, run_id=RUN_ID, batch_size=BATCH_SIZE, max_rounds=MAX_ROUNDS,
    rounds_run=n_rounds, confidence_margin=CONFIDENCE_MARGIN, relabel_pool=RELABEL_POOL, seed=SEED,
    base_choice=choice, base_selection_sha256=digest(MODEL_DIR/'selection_manifest.json'),
    unlabeled_source_sha256=source.sha256, split_sha256=digest(DATA_DIR/'splits/split_assignments.csv'),
    round0_reproduced=True, test_used_for_training=False, python=platform.python_version(), sklearn=sklearn.__version__,
    artifacts=sorted(p.name for p in OUT.iterdir() if p.is_file())))